# Exercise 4: Validation and Cross-Validation

```{admonition} Run or download this notebook
:class: how-to-use

This page is fully readable as it is, and the interactive activities work
here in the browser. To **run or change the Python code**, use the portable
version of the notebook:

- **[Open the portable notebook in Colab](https://colab.research.google.com/github/yoavmp/ml-neuro-tutorials/blob/main/book/downloads/chapter_04/exercise_04_portable.ipynb)**
- **[View or download the portable `.ipynb`](https://raw.githubusercontent.com/yoavmp/ml-neuro-tutorials/main/book/downloads/chapter_04/exercise_04_portable.ipynb)**
  for VS Code or Jupyter

The portable notebook keeps every Python analysis cell and swaps each
embedded activity for a link back to this page.
```

## What this notebook covers

This is Exercise 4 of *Machine Learning for Neuroscience*. Exercise 4 is about
**validation and cross-validation** -- comparing a single train/test split with
cross-validation, and using training, validation, and test data to tune and
honestly evaluate a model.

It reuses Exercise 2's ABIDE-II age-prediction task and its fixed KNN
predictor set unchanged. The predictors themselves were already fixed before
this lesson -- nothing here selects, adds, or removes a feature. Feature
selection is Exercise 5's topic, not this one.

In this notebook you will:

1. reuse Exercise 2's ABIDE age-prediction data and its fixed KNN predictors;
2. compare a single train/test split with cross-validation across several
   sample sizes;
3. evaluate one fixed KNN model with 5-fold cross-validation;
4. distinguish training, validation, and test data;
5. tune KNN's number of neighbours, `k`, using training and validation data
   only, then reveal a test result once;
6. use nested cross-validation to keep tuning separate from evaluation;
7. compare validation strategies for different research situations.

**Prerequisites:** Exercise 2's regression and KNN material; comfort with
`pandas`, `numpy`, and the scikit-learn `fit` / `predict` / `Pipeline`
pattern.

## 1. Our Regression Data

This exercise reuses Exercise 2's ABIDE-II regression table unchanged: the
same participants, the same target, and the same fixed predictor set. Nothing
in this section is new -- it is here so this notebook is self-contained.

In [1]:
# Data loading. In the published book this cell is collapsed; it is plain,
# runnable Python -- one public CSV pinned to an immutable commit. Nothing
# here is repository-specific. This is the exact same ABIDE-II table Exercise
# 2 uses, with age as a native column.
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

PIN = "e4eed3c4daa7f40b0ba931182a8c7e5e691dba6b"
BASE = f"https://raw.githubusercontent.com/neurohackademy/nh2020-curriculum/{PIN}/tu-machine-learning-yarkoni/data"

model_df = pd.read_csv(f"{BASE}/abide2.tsv", sep="\t")
model_df = model_df.loc[:, [c for c in model_df.columns if not str(c).startswith("Unnamed")]]

BRAIN_COLS = [c for c in model_df.columns if c.startswith("fs")]
print(f"data table: {model_df.shape[0]} participants x {model_df.shape[1]} columns")

data table: 1004 participants x 1446 columns


In [2]:
# The same fixed 360 cortical-thickness predictors used in Exercise 2's KNN
# worked example (fsCT_*, both hemispheres of every HCP-MMP1 parcel). This
# exercise is about how we EVALUATE and TUNE a model, not about choosing
# features -- feature selection is Exercise 5's topic.
FEATURES = [c for c in BRAIN_COLS if c.startswith("fsCT_")]
assert all(c.startswith("fsCT_") for c in FEATURES)           # brain-only
assert "age" not in FEATURES                                  # no target leakage
print(f"{len(FEATURES)} predictors, fixed before this lesson, e.g. {FEATURES[:3]}")

360 predictors, fixed before this lesson, e.g. ['fsCT_L_V1_ROI', 'fsCT_L_MST_ROI', 'fsCT_L_V6_ROI']


In [3]:
# 1-2. brain-only X and target y; reusing the exact participant filter,
# feature order, and stratified train/test split from Exercise 2, so the
# training and test partitions below are IDENTICAL to Exercise 2's.
has_age = model_df["age"].notna()
X = model_df.loc[has_age, FEATURES].to_numpy(float)
y = model_df.loc[has_age, "age"].to_numpy(float)
groups = model_df.loc[has_age, "group"].to_numpy()      # 1 = autism, 2 = control

X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.25, random_state=42, stratify=groups
)
print(f"n_train = {len(y_train)}   n_test = {len(y_test)}   n_features = {X.shape[1]}")

n_train = 753   n_test = 251   n_features = 360


A short reminder before we start, both already established in Exercise 2:

- KNN predicts a participant's age by averaging the `k` training participants
  whose **predictors** are closest to theirs, so it depends directly on
  distances in predictor space;
- an unscaled predictor with a large numeric range would dominate that
  distance just because of its units. A scikit-learn `Pipeline` keeps
  `StandardScaler` bundled with the model, so scaling is learned from
  training data only and refit separately inside every fold below -- never
  from validation or test data.

## 2. From One Split to Cross-Validation

Exercises 1-3 evaluated every model with one train/test split. That split is
not wrong: it is a genuine, honest estimate of performance on participants the
model never saw while fitting. But its exact number depends on *which*
participants happened to land in the training set and which landed in the
test set -- and that dependence matters more when the available sample is
small.

**Cross-validation** partitions the data into several folds, fits the model
on all-but-one fold, evaluates it on the held-out fold, and repeats until
every fold has served as the held-out one once. That gives several
performance estimates from the same data instead of one.

With the full eligible cohort (1004 participants), a single 75/25 split and
5-fold cross-validation give broadly similar answers across five predeclared
random seeds. That is not true at every sample size: at 30 and 50
participants, the single-split test R² swings widely across seeds -- for some
seeds it even lands below zero, worse than always predicting the
training-set mean age. Try the activity below to see this directly.

> The full ABIDE sample is relatively large for this demonstration. The
> small-sample settings represent research situations in which only tens of
> participants are available.

<iframe
  title="Interactive single-split and cross-validation stability comparison for predicting age from brain structure"
  src="../../_static/widgets/app/index.html?config=../configs/validation_stability.json"
  loading="lazy"
  width="100%"
  height="1750"
  class="ml-activity"
  style="width: 100%;"
></iframe>

## 3. Cross-Validation for a Fixed KNN Model

The activity above lets you compare a single split with cross-validation
across several sample sizes and seeds. The cell below runs one specific
cross-validation directly in Python, on the full eligible cohort, with `k`
fixed at the same value Exercise 2 used -- this section evaluates one model,
it does not choose `k`.

Cross-validation with `K` folds:

1. split the participants into `K` folds;
2. fit on `K - 1` folds;
3. evaluate on the remaining fold;
4. repeat until every fold has been used for evaluation exactly once;
5. summarize the `K` fold results.

scikit-learn's `cross_validate` reports **negative** mean squared error
(`neg_mean_squared_error`) so that, like every other scorer it ranks, a
higher score is always better. The cell below negates it back to ordinary
(positive) MSE before printing -- lower MSE is better.

In [4]:
# Ordinary, editable Python: the same kind of 5-fold cross-validation the
# activity above lets you explore, computed directly with scikit-learn on the
# full eligible cohort, with k fixed throughout -- this cell evaluates one
# model, it does not choose k.
K_EXAMPLE = 20  # the same fixed value used in Exercise 2's KNN worked example

cv_pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=K_EXAMPLE))
cv_folds = KFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_validate(
    cv_pipe, X, y, cv=cv_folds, scoring=("neg_mean_squared_error", "r2")
)

# scikit-learn reports NEGATIVE MSE; negate it back to ordinary (positive) MSE.
cv_fold_mse = -cv_scores["test_neg_mean_squared_error"]
cv_fold_r2 = cv_scores["test_r2"]
for i, (mse, r2) in enumerate(zip(cv_fold_mse, cv_fold_r2), start=1):
    print(f"fold {i}: MSE = {mse:.1f}   R^2 = {r2:.3f}")
print(f"mean MSE = {cv_fold_mse.mean():.1f}   (sd {cv_fold_mse.std():.1f})")
print(f"mean R^2 = {cv_fold_r2.mean():.3f}")

fold 1: MSE = 42.9   R^2 = 0.554
fold 2: MSE = 33.4   R^2 = 0.596
fold 3: MSE = 31.5   R^2 = 0.554
fold 4: MSE = 34.9   R^2 = 0.662
fold 5: MSE = 26.3   R^2 = 0.709
mean MSE = 33.8   (sd 5.4)
mean R^2 = 0.615


A few things to notice:

- five folds give five MSE values from the *same* 1004 participants and the
  *same* fixed `k = 20` -- they differ because each fold holds out a
  different subset of participants, not because the model changed;
- the mean and standard deviation summarize that spread in one number each;
  a small standard deviation here means the folds mostly agree;
- this is model **evaluation**, not model **tuning** -- `k` stayed fixed
  throughout. Section 5 introduces tuning.

## 4. Train, Validation, and Test Data

So far every model in this course has used one setting chosen in advance --
Exercise 2's `k = 20`, Exercise 3's `C = 1.0`. A **hyperparameter** is a model
setting chosen before fitting; for KNN, the number of neighbours `k` is a
hyperparameter. When several candidate values are genuinely worth comparing,
three data roles keep that comparison honest:

| Subset          | Purpose                              |
| ---------------- | ------------------------------------ |
| Training data   | Fit model parameters                 |
| Validation data | Compare hyperparameter values        |
| Test data       | Evaluate the selected procedure once |

The procedure:

1. fit candidate KNN models, one per candidate `k`, on training data;
2. compare their validation MSE;
3. choose `k` using validation data only;
4. refit using training and validation data together;
5. evaluate once on untouched test data.

Looking at the test result repeatedly -- checking it, changing `k`, checking
again -- turns the test set into another validation set: the number it
eventually reports is no longer an honest estimate of performance on
participants the whole procedure never touched.

```{admonition} Think first
:class: think-first
Two values of `k` matter for what follows: the **training-selected `k`** is
the candidate with the lowest *training* MSE; the **validation-selected `k`**
is the candidate with the lowest *validation* MSE.

1. Which one do you expect to perform best on the training set itself: the
   training-selected `k` or the validation-selected `k`?
2. Which one do you expect to perform better on unseen test participants?
3. Why might the value that gives the lowest training error fail on new
   participants?
4. Should you change `k` after seeing the test result?
```

## 5. Tune KNN Without Looking at the Test Set

The activity below plots training and validation MSE across candidate `k`
values side by side. Choose a final `k`, then press "Lock Choice and Reveal
Test Result" to add a third panel: test MSE across the same candidates,
revealed for the first and only time.

<iframe
  title="Interactive KNN tuning activity: choose k before revealing the test result"
  src="../../_static/widgets/app/index.html?config=../configs/validation_lock_test.json"
  loading="lazy"
  width="100%"
  height="1400"
  class="ml-activity"
  style="width: 100%;"
></iframe>

Once you lock a choice above, a third panel appears: test MSE across the
same candidate `k` values, revealed for the first time. Look at its shape
next to the other two -- the test curve tracks the validation curve far
more closely than the training curve: both dip to a minimum somewhere in
the middle of the candidate range, while training MSE keeps falling all
the way to `k = 1`. That resemblance -- not a guarantee that the
validation-selected value always wins -- is this section's central
lesson: validation error is a far better guide to test performance than
training error is, even on a single split.

```{admonition} Think again, after revealing
:class: think-first
1. Which curve does the revealed test-MSE curve resemble more: the
   training curve or the validation curve?
2. Did the validation-selected `k` also minimize test MSE on this
   particular split?
3. If a different `k` looks better on the revealed test curve, why must
   you not switch to it now?
4. What would happen to the meaning of the test set if you repeated this
   whole procedure -- lock, reveal, reconsider -- many times?
```

### Optional: Reproduce the Tuning Activity in Python

The interactive activity above contains the main lesson. Expand this
optional section if you want to reproduce the analysis in Python.

In [5]:
# Further split the OUTER training partition set aside in Section 1 (753
# participants) into a fitting subset and a validation subset. The outer test
# set (X_test, y_test) is not read anywhere in this cell.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=7, stratify=groups_train
)
print(f"n_fit = {len(y_fit)}   n_val = {len(y_val)}")

CANDIDATE_KS = [1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 50, 75, 100, 150, 250, 400, 564]

tuning_rows = []
for k in CANDIDATE_KS:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    pipe.fit(X_fit, y_fit)
    train_mse = mean_squared_error(y_fit, pipe.predict(X_fit))
    val_mse = mean_squared_error(y_val, pipe.predict(X_val))
    tuning_rows.append({"k": k, "train_mse": train_mse, "val_mse": val_mse})

tuning = pd.DataFrame(tuning_rows)
training_selected_k = int(tuning.loc[tuning["train_mse"].idxmin(), "k"])
validation_selected_k = int(tuning.loc[tuning["val_mse"].idxmin(), "k"])
print(f"training-selected k   = {training_selected_k}")
print(f"validation-selected k = {validation_selected_k}")
tuning.round(2)

n_fit = 564   n_val = 189


training-selected k   = 1
validation-selected k = 25


,k,train_mse,val_mse
0,1,0.00,51.91
1,2,14.52,36.43
2,3,19.98,32.12
3,5,26.75,33.66
4,8,31.84,30.28
5,10,32.45,29.03
6,15,34.57,26.98
7,20,37.13,27.13
8,25,38.33,26.66
9,30,39.68,28.41


Run the next cell only after you have chosen (or confirmed) a `k` using only
the two columns above. It reveals the test-set result for the
training-selected value, the validation-selected value, and your own choice
-- once, matching the "Lock Choice and Reveal Test Result" button above.

In [6]:
# Reveal, once: refit on the full outer-training partition (fit + validation
# combined) at each of the values below, then score on the outer test set --
# looked at here for the first and only time in this notebook. Edit
# STUDENT_CHOICE_K to try a different value, then rerun this cell.
STUDENT_CHOICE_K = validation_selected_k

for label, k in [
    ("training-selected", training_selected_k),
    ("validation-selected", validation_selected_k),
    ("your choice", STUDENT_CHOICE_K),
]:
    pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    test_mse = mean_squared_error(y_test, pred)
    test_r2 = r2_score(y_test, pred)
    print(f"{label:20s} k = {k:>3d}   test MSE = {test_mse:.1f}   test R^2 = {test_r2:.3f}")

training-selected    k =   1   test MSE = 50.2   test R^2 = 0.462
validation-selected  k =  25   test MSE = 32.6   test R^2 = 0.651
your choice          k =  25   test MSE = 32.6   test R^2 = 0.651


A few things to hold onto:

- the training-selected value is expected to look best on training data
  *because it was selected there* -- `k = 1` fits every training participant
  almost perfectly and generalises poorly;
- the validation-selected value is expected to generalise better, since
  validation participants were never used to fit any candidate model;
- "expected" is not "guaranteed": a single test split can occasionally favour
  a different value by chance;
- whichever value you chose above, it must not change after seeing this
  cell's output. In a real analysis, this cell would run once.

## 6. Nested Cross-Validation

Section 4-5 used one train/validation/test split to tune and evaluate `k`.
With limited data, a single split spends participants on three separate
roles and reports only one evaluation. **Nested cross-validation** reuses the
same participants for both tuning and evaluation, without letting the two
mix:

- an **inner** cross-validation chooses `k`, using only the current outer
  fold's training data;
- an **outer** cross-validation evaluates the complete tuning procedure --
  inner selection included -- on data the inner loop never saw.

<div class="ml-ncv-diagram" role="group" aria-label="Diagram: outer cross-validation rotates an outer-test fold across 5 iterations; for one outer-training set, inner cross-validation rotates an inner-validation fold across 5 iterations to choose k, which is then refit on all outer-training data and evaluated once on that iteration's outer-test fold." style="margin:1.4rem 0;">
<div style="display:flex;flex-wrap:wrap;gap:4px 4px;align-items:center;margin-bottom:12px;font-size:0.86rem;"><span style="display:inline-flex;align-items:center;gap:6px;margin-right:16px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span>Outer training</span></span><span style="display:inline-flex;align-items:center;gap:6px;margin-right:16px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span><span>Outer test</span></span><span style="display:inline-flex;align-items:center;gap:6px;margin-right:16px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span>Inner training</span></span><span style="display:inline-flex;align-items:center;gap:6px;margin-right:16px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span><span>Inner validation</span></span></div>
<div style="display:flex;overflow-x:auto;gap:20px;align-items:flex-start;padding-bottom:6px;" aria-hidden="true">
<div style="flex:0 0 auto;">
<div style="font-weight:600;font-size:0.92rem;margin-bottom:2px;">Outer cross-validation</div>
<div style="font-size:0.78rem;color:var(--ml-muted, #5c5674);margin-bottom:8px;max-width:180px;">5 outer iterations; the outer-test segment (yellow) rotates. Outlined row: the iteration zoomed into on the right.</div>
<div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">1</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">2</span><span style="display:flex;gap:3px;outline:2px solid var(--ml-accent, #dd5f1b);outline-offset:2px;border-radius:4px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">3</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">4</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">5</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-train, #a9c2e3);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-outer-test, #e6c368);"></span></span></div>
</div>
<div style="flex:0 0 auto;align-self:center;font-size:1.5rem;padding-top:44px;color:var(--ml-muted, #5c5674);">&#8594;</div>
<div style="flex:0 0 auto;">
<div style="font-weight:600;font-size:0.92rem;margin-bottom:2px;">Inner cross-validation</div>
<div style="font-size:0.78rem;color:var(--ml-muted, #5c5674);margin-bottom:8px;max-width:180px;">Zoomed from outer iteration 2's training data: 5 inner iterations; the inner-validation segment (orange) rotates.</div>
<div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">1</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">2</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">3</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">4</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span></span></div><div style="display:flex;align-items:center;gap:5px;margin-bottom:3px;"><span style="width:15px;font-size:0.72rem;color:var(--ml-muted, #5c5674);text-align:right;">5</span><span style="display:flex;gap:3px;"><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-train, #a3cbae);"></span><span style="display:inline-block;width:24px;height:20px;border-radius:3px;background-color:var(--ml-ncv-inner-val, #e0a877);"></span></span></div>
</div>
<div style="flex:0 0 auto;align-self:center;font-size:1.5rem;padding-top:44px;color:var(--ml-muted, #5c5674);">&#8594;</div>
<div style="flex:0 0 auto;display:flex;flex-direction:column;gap:12px;max-width:190px;padding-top:30px;">
<div style="border-left:4px solid var(--ml-ncv-inner-val, #e0a877);padding-left:8px;font-size:0.86rem;">Choose <code>k</code></div>
<div style="border-left:4px solid var(--ml-ink, #1b1826);padding-left:8px;font-size:0.86rem;">Refit the selected <code>k</code> on all outer-training data</div>
<div style="border-left:4px solid var(--ml-ncv-outer-test, #e6c368);padding-left:8px;font-size:0.86rem;">Evaluate the tuning procedure</div>
</div>
</div>
<p style="margin-top:10px;font-size:0.86rem;color:var(--ml-muted, #5c5674);">Inner-validation folds (orange) choose <code>k</code>, using only the current outer iteration's training data. Outer-test folds (yellow) then evaluate the complete tuning procedure -- inner selection included -- on data the inner loop never saw. Outer-test folds never choose <code>k</code>.</p>
</div>

The outer test fold must not participate in scaling, choosing `k`, comparing
candidate models, or any other preprocessing decision for that fold. Because
each outer fold's training data contains different participants, different
outer folds *can* select different values of `k` -- the inner
cross-validation does not have to agree with itself from fold to fold.

In [7]:
# One complete nested-cross-validation pipeline: an inner GridSearchCV
# chooses k using only this outer fold's training data; the outer fold then
# evaluates that choice on data the inner loop never saw. Scaling is fitted
# inside the pipeline, so it is refit separately for every inner and outer
# training partition.
NESTED_CANDIDATE_KS = [8, 10, 12, 15, 18, 20, 22, 25, 28, 30, 40, 50]
N_OUTER, N_INNER = 5, 5
OUTER_SEED, INNER_SEED = 100, 101

outer_cv = KFold(n_splits=N_OUTER, shuffle=True, random_state=OUTER_SEED)
inner_cv = KFold(n_splits=N_INNER, shuffle=True, random_state=INNER_SEED)

nested_rows = []
for fold_i, (train_idx, test_idx) in enumerate(outer_cv.split(X)):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    pipe = Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())])
    grid = GridSearchCV(
        pipe,
        param_grid={"knn__n_neighbors": NESTED_CANDIDATE_KS},
        scoring="neg_mean_squared_error",
        cv=inner_cv,
    )
    grid.fit(X_tr, y_tr)  # only this outer fold's TRAINING data
    selected_k = grid.best_params_["knn__n_neighbors"]

    pred = grid.predict(X_te)  # the outer TEST fold, evaluated once
    nested_rows.append(
        {
            "outer_fold": fold_i + 1,
            "selected_k": selected_k,
            "n_train": len(y_tr),
            "n_test": len(y_te),
            "outer_test_mse": mean_squared_error(y_te, pred),
            "outer_test_r2": r2_score(y_te, pred),
        }
    )

nested = pd.DataFrame(nested_rows)
nested.round(3)

,outer_fold,selected_k,n_train,n_test,outer_test_mse,outer_test_r2
0,1,15,803,201,36.265,0.678
1,2,12,803,201,31.147,0.565
2,3,10,803,201,43.106,0.605
3,4,18,803,201,20.459,0.646
4,5,15,804,200,35.798,0.605


In [8]:
print(f"mean outer-test MSE = {nested['outer_test_mse'].mean():.1f}   (sd {nested['outer_test_mse'].std(ddof=0):.1f})")
print(f"mean outer-test R^2 = {nested['outer_test_r2'].mean():.3f}")
print(f"selected k per outer fold: {list(nested['selected_k'])}")

mean outer-test MSE = 33.4   (sd 7.5)
mean outer-test R^2 = 0.620
selected k per outer fold: [15, 12, 10, 18, 15]


With this denser grid, the outer folds no longer all agree: the
`selected_k` column above shows four different values across the five
outer folds. Different outer folds *can* select different `k` values,
because each outer training set contains different participants -- that
is what happened here. They are not guaranteed to disagree, only
permitted to: if a rerun's folds all happened to agree instead, that
would be an equally valid outcome, not evidence against the method. The
best *inner* cross-validation score is not reported as a final
performance estimate: it is biased toward whichever candidate happened
to look best on the data used to choose it. The **outer**-test MSE and
R² above -- averaged over five folds that never influenced any tuning
decision -- estimate the performance of the whole tuning procedure.

The activity below lets you inspect this fold by fold: which candidate
`k` values the inner cross-validation compared, which one it selected,
and how that selection then scored on the outer test fold.

<iframe
  title="Interactive nested cross-validation explorer for predicting age from brain structure"
  src="../../_static/widgets/app/index.html?config=../configs/nested_cv_explorer.json"
  loading="lazy"
  width="100%"
  height="1350"
  class="ml-activity"
  style="width: 100%;"
></iframe>

## 7. Choosing a Validation Strategy

| Situation                                 | Appropriate approach               |
| ------------------------------------------ | ----------------------------------- |
| Large dataset and fixed model             | Train/test split may be sufficient |
| Tune a model with enough data             | Train/validation/test split        |
| Evaluate a fixed model with limited data  | Cross-validation                   |
| Tune and evaluate with limited data       | Nested cross-validation            |

Nested cross-validation estimates the performance of the complete tuning
procedure. After evaluation, a final model can be tuned and fitted using all
available development data, but its performance estimate still comes from
the outer folds.

In the homework, you will apply the same workflow to logistic-regression
classification.

These examples treat participants as independent observations. Repeated
measurements from the same participant would need to remain together in the
same fold.

## In summary

- One train/test split is an honest but single estimate; its exact number
  depends on which participants land in each subset, and that dependence is
  most visible at small sample sizes. Cross-validation evaluates the same
  fixed model across several partitions instead of one.
- Training, validation, and test data play three different roles: fit
  parameters, compare hyperparameter candidates, and evaluate the selected
  procedure once. Reusing the test set to choose or re-choose a hyperparameter
  turns it into another validation set.
- The training-selected `k` and the validation-selected `k` are different
  values here (Section 5) for a concrete reason: the training-selected value
  was chosen using the same data it is then judged on.
- Nested cross-validation keeps tuning and evaluation separate without
  spending participants on three fixed roles: an inner loop chooses `k`, an
  outer loop evaluates the whole procedure on data the inner loop never saw.
  Different outer folds can select different `k` values, because each has
  different training data.
- No single validation strategy is always correct -- the right one depends on
  dataset size and on whether the model is being tuned or only evaluated.

Next practice: regularization and feature selection.

### Questions to take away

1. Why does a single train/test split's result depend on which participants
   land in the test set, and why does that dependence matter more at small
   sample sizes?
2. In Section 3, why does scikit-learn report *negative* mean squared error,
   and why does that not change which fold's model performed best?
3. What is a hyperparameter? Give one example from this notebook and one from
   Exercise 3.
4. In Section 5, why is the training-selected `k` expected to look better than
   the validation-selected `k` on the training data itself?
5. Why would repeatedly checking the test result and re-choosing `k`
   invalidate the test score as an evaluation?
6. In Section 6, what does the inner cross-validation choose, and what does
   the outer cross-validation evaluate?
7. Why is the best *inner* cross-validation score not reported as the final
   performance estimate?
8. Why can different outer folds select different values of `k`?
9. A dataset has repeated scans from the same participants. What would need
   to change about how folds are constructed?